# Reinforcement Learning for Warranty Diagnostics — Theory, Math & CUDA Playbook

**Domain:** WarrantyGraph / remote diagnosis (GraphRAG + optional vision OCR)
**Package:** `ml/fault_code_rl/`
**Adjoins:**
- [`fault_code_gan_synthetic_images.ipynb`](./fault_code_gan_synthetic_images.ipynb) — synthetic fault-code images
- [`fault_code_vision_mlops_playbook.ipynb`](./fault_code_vision_mlops_playbook.ipynb) — CUDA vision MLOps

---

## Client question this notebook answers

> *If reinforcement learning is required, what kind, with what mathematics, how does it work with GraphRAG, and how do we train on CUDA like the vision stack?*

### Honest product stance

| Layer | Role of RL |
|-------|------------|
| **GraphRAG (Neo4j)** | **Primary reasoner** — FailureMode ranking, CONFIRMS steps, provenance (keep) |
| **Vision OCR** | Input normalizer: photo → error code |
| **RL (optional)** | **Decision policy** on top: next-best step, escalation cost trade-offs, offline improvement from claim outcomes |

**Do not** replace deterministic graph diagnosis with unconstrained RL.
**Do** use RL to re-rank **graph-eligible** actions under a logged reward (resolve / cost / reopen).

## Part 0 — Which RL for *this* project?

| Need | RL family | Why it fits warranty diagnostics | Hardware |
|------|-----------|----------------------------------|----------|
| Next diagnostic step / question | **Contextual bandit** (LinUCB, Thompson) | Partial feedback; one action per decision; safer | CPU |
| Multi-step session (tests in sequence) | **MDP + Q-learning / DQN** | State = evidence so far; delayed resolve reward | Q: CPU; DQN: **CUDA** preferred |
| Learn only from historical claims | **Offline / batch RL + IPS** | No live random exploration on customers | CPU / CUDA for deep models |
| Escalation vs self-serve cost | Bandit or constrained MDP | Explicit cost-sensitive rewards | CPU |
| Replace GraphRAG | **Avoid** | Audit, safety, sparse/noisy rewards | — |

### Recommended adoption order (client roadmap)

1. **Log** (context, suggested step, outcome, propensity)
2. **Contextual bandit** in **shadow mode** (log only)
3. **Offline eval** (IPS) → canary
4. **MDP/DQN** only when multi-turn sessions + enough data

Authoritative algorithms in this notebook: UCB1 (Auer et al.), LinUCB (Li et al. 2010), Q-learning (Watkins; Sutton & Barto), DQN (Mnih et al. 2015).

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path(".").resolve()
if not (ROOT / "ml").is_dir():
    if (ROOT.parent / "ml").is_dir():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from ml.fault_code_rl.device import device_report, pick_device
from ml.fault_code_rl.domain import STEP_IDS, FAILURE_MODES, ERROR_CODE_PRIOR
from ml.fault_code_rl.mdp import DiagnosticMDP
from ml.fault_code_rl.bandits import EpsilonGreedy, UCB1, LinUCB, ThompsonSampling, run_bandit_on_mdp
from ml.fault_code_rl.q_learning import QLearningConfig, train_q_learning, greedy_policy_steps
from ml.fault_code_rl.dqn import DQNConfig, train_dqn, dqn_act
from ml.fault_code_rl.offline import behavior_log_from_random_policy, inverse_propensity_score, safety_mask_graph_eligible, masked_argmax
from ml.fault_code_rl.pipeline import rl_when_needed, mlops_checklist, integration_sketch
from ml.fault_code_rl.rewards import DEFAULT_REWARD
import pandas as pd
import torch

ART = ROOT / "notebooks" / "fault_code_rl_artifacts"
ART.mkdir(parents=True, exist_ok=True)

print("ROOT", ROOT)
print("device_report:", json.dumps(device_report(), indent=2))
print("device", pick_device())
print("Actions (diagnostic steps):", STEP_IDS)
print("Failure modes:", [f.fm_id for f in FAILURE_MODES])
print("Reward config:", DEFAULT_REWARD)

## Part 1 — Mathematics: diagnosis as an MDP

A **Markov Decision Process** is \(\langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle\).

| Symbol | In this product |
|--------|-----------------|
| State \(s \in \mathcal{S}\) | Evidence so far: steps done, product, error code, symptoms |
| Action \(a \in \mathcal{A}\) | Next diagnostic step / question / escalate |
| Transition \(P(s'|s,a)\) | Test outcome (pass/fail evidence) changes belief state |
| Reward \(R(s,a)\) | \(+\) resolve success, \(-\) step cost, \(-\) wrong path / reopen |
| Discount \(\gamma \in [0,1)\) | Prefer faster resolution |

### Objective

\[
\pi^\star = \arg\max_\pi \; \mathbb{E}_{\pi}\Big[\sum_{t=0}^{T} \gamma^t r_t\Big]
\]

### Bellman optimality (Q-function)

\[
Q^\star(s,a) = \mathbb{E}\big[r + \gamma \max_{a'} Q^\star(s',a') \mid s,a\big]
\]

**Tabular Q-learning update** (Watkins):

\[
Q(s,a) \leftarrow Q(s,a) + \alpha \big(r + \gamma \max_{a'} Q(s',a') - Q(s,a)\big)
\]

### Constraint (enterprise)

\[
a_t \in \mathcal{A}_{\text{eligible}}(s_t) \subseteq \text{GraphRAG CONFIRMS candidates}
\]

RL **must not** invent steps or codes outside the knowledge graph.

## Part 2 — Mathematics: contextual bandits (next-best step)

When each decision is “pick one step now” with delayed outcome, a **contextual bandit** is enough:

\[
a_t = \pi(x_t), \quad r_t \sim R(\cdot \mid x_t, a_t)
\]

No long-horizon \(P(s'|s,a)\) model required.

### UCB1 (non-contextual)

\[
a_t = \arg\max_a \left( \hat\mu_a + c \sqrt{\frac{\ln t}{n_a}} \right)
\]

### LinUCB (contextual)

For context \(x \in \mathbb{R}^d\), per-arm ridge regression \(\hat\theta_a\):

\[
a_t = \arg\max_a \left( \hat\theta_a^\top x + \alpha \sqrt{x^\top A_a^{-1} x} \right)
\]

### Why bandits first for WarrantyGraph

- Maps cleanly onto “next CONFIRMS step”
- Exploration is controllable
- Works with sparse claim rewards
- CPU-only; no CUDA required

## Part 3 — Working simulator (`DiagnosticMDP`)

Hidden true failure mode is sampled from error-code priors (like `INDICATES` boosts).
Actions are diagnostic steps with costs. Positive evidence on confirming steps yields resolution reward.

In [ ]:
env = DiagnosticMDP(max_steps=4, seed=7)
s = env.reset()
print("context error_code=", env.context.error_code, "true_fm=", env.true_fm)
print("symptoms", env.context.symptom_flags)
print("feature dim", len(env.encode_state_features()))

# manual short episode
done = False
total = 0.0
trace = []
while not done:
    # naive: prefer filter then hose then pump
    prefer = ["s_filter", "s_hose", "s_pump", "s_balance", "s_inlet", "s_door", "s_escalate"]
    a = next(STEP_IDS.index(x) for x in prefer if not (env.done_mask & (1 << STEP_IDS.index(x))))
    tr = env.step(a)
    total += tr.reward
    trace.append((STEP_IDS[a], round(tr.reward, 2), tr.info.get("evidence_positive"), tr.info.get("success")))
    done = tr.done
print("trace:", trace)
print("episode return", round(total, 2))

print("\nWhen is RL needed? (from package)")
display_df = pd.DataFrame(rl_when_needed())
print(display_df.to_string(index=False))

## Part 4 — Working code: bandits for first diagnostic action

We optimize the **first** step recommendation; remainder of the episode is exploratory noise so the bandit reward is a noisy Monte Carlo return (standard bandit-with-complex-reward demo).

In [ ]:
N = 600
d = len(DiagnosticMDP(seed=0).context.feature_vector())
curves = {}
final = {}
for name, make, contextual in [
    ("epsilon_greedy", lambda: EpsilonGreedy(len(STEP_IDS), epsilon=0.1, seed=0), False),
    ("ucb1", lambda: UCB1(len(STEP_IDS), c=1.5), False),
    ("thompson", lambda: ThompsonSampling(len(STEP_IDS), seed=0), False),
    ("linucb", lambda: LinUCB(len(STEP_IDS), d=d, alpha=0.5), True),
]:
    env = DiagnosticMDP(seed=0)
    out = run_bandit_on_mdp(env, make(), n_episodes=N, contextual=contextual)
    curves[name] = out["curve"]
    final[name] = out["mean_return"]
    print(f"{name:16s} mean_return={out['mean_return']:.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
for name, c in curves.items():
    ax.plot(c, label=name, alpha=0.9)
ax.set_xlabel("episode")
ax.set_ylabel("avg return so far")
ax.set_title("Bandit learning curves (first-step policy on DiagnosticMDP)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ART / "bandit_curves.png", dpi=120)
plt.show()
(ART / "bandit_results.json").write_text(json.dumps(final, indent=2))

## Part 5 — Working code: tabular Q-learning (sequential policy)

Full multi-step policy over bitmask states. No GPU needed. Good for client *theory → working agent* demo.

In [ ]:
q_result = train_q_learning(cfg=QLearningConfig(episodes=1200, seed=42))
print("mean return last 100:", q_result["mean_return_last_100"])
print("success rate last 100:", q_result["success_rate_last_100"])

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(q_result["moving_avg_return"], color="steelblue")
ax.set_title("Q-learning moving average return (window=100)")
ax.set_xlabel("episode")
ax.set_ylabel("return")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ART / "q_learning_curve.png", dpi=120)
plt.show()

# Greedy action sequence from empty evidence
acts = greedy_policy_steps(q_result["agent"], state=0, max_steps=4)
print("Greedy steps from empty state:", [STEP_IDS[a] for a in acts])

np.savez(ART / "q_policy.npz", Q=q_result["Q"])
print("saved", ART / "q_policy.npz")

## Part 6 — Working code: DQN + CUDA/MPS (same hardware story as vision)

**Deep Q-Network** approximates \(Q(s,a;\theta)\) with a neural net (Mnih et al., Nature 2015):

- Replay buffer + target network
- ε-greedy exploration
- Trains on `pick_device()` → **CUDA** (client), **MPS** (Mac demo), or CPU

```bash
# Client GPU (shared image with vision)
docker build -f docker/Dockerfile.ml -t warrantygraph-ml:latest .
docker run --gpus all -v "$PWD":/workspace -w /workspace warrantygraph-ml:latest \
  python -m ml.fault_code_rl.train --algo dqn --episodes 800 --require-cuda \
  --out models/artifacts/rl/dqn.pt
```

In [ ]:
dqn_cfg = DQNConfig(episodes=400, seed=42, require_cuda=False)  # True on client NVIDIA
dqn_result = train_dqn(dqn_cfg)
print("device:", dqn_result["device"])
print("mean return last 100:", dqn_result["mean_return_last_100"])

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
# smooth returns
w = 20
ret = np.array(dqn_result["returns"])
smooth = np.convolve(ret, np.ones(w)/w, mode="valid")
ax[0].plot(smooth)
ax[0].set_title("DQN episode return (smoothed)")
ax[0].grid(True, alpha=0.3)
if dqn_result["losses"]:
    ax[1].plot(dqn_result["losses"][:: max(1, len(dqn_result["losses"])//500)])
    ax[1].set_title("DQN TD loss (subsampled)")
    ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ART / "dqn_curves.png", dpi=120)
plt.show()

ckpt = ART / "dqn_policy.pt"
torch.save(
    {
        "policy": dqn_result["policy"].state_dict(),
        "obs_dim": dqn_result["obs_dim"],
        "n_actions": dqn_result["n_actions"],
        "steps": STEP_IDS,
        "device_report": dqn_result["device_report"],
    },
    ckpt,
)
print("saved", ckpt)

## Part 7 — Offline RL & safety (what clients actually need)

### Inverse Propensity Scoring (sketch)

Behavior policy \(\pi_b\) logged propensities \(\pi_b(a|x)\). For target \(\pi\):

\[
\hat V_{\mathrm{IPS}} = \frac{1}{N}\sum_{i=1}^{N} \frac{\mathbf{1}[a_i=\pi(x_i)]}{\pi_b(a_i|x_i)} r_i
\]

### Production pattern

1. **Shadow:** serve GraphRAG; log what RL *would* have done
2. **Offline IPS / DR** on logged data
3. **Canary** small %
4. **Action mask** = CONFIRMS-eligible steps only

In [ ]:
env = DiagnosticMDP(seed=1)
logs = behavior_log_from_random_policy(env, n=800, seed=1)
print("logged decisions", len(logs), "example reward", round(logs[0].reward, 2))

# Target policy: always pick action 0 (filter) — IPS estimate
ips_always_0 = inverse_propensity_score(logs, target_actions=[0] * len(logs))
# Target = behavior clone
ips_behavior = inverse_propensity_score(logs, target_actions=[row.action for row in logs])
print("IPS always-arm0:", round(ips_always_0, 3))
print("IPS behavior-clone (should ~ mean reward):", round(ips_behavior, 3),
      "empirical mean", round(float(np.mean([r.reward for r in logs])), 3))

# Safety mask example: only graph-eligible steps for a drain-heavy case
eligible = [STEP_IDS.index(s) for s in ("s_filter", "s_hose", "s_pump", "s_escalate")]
mask = safety_mask_graph_eligible(len(STEP_IDS), eligible)
q_fake = np.array([0.1, 0.9, 0.2, 5.0, 0.3, 0.4, 0.0])  # inlet looks best raw
print("unmasked argmax", STEP_IDS[int(np.argmax(q_fake))])
print("masked argmax  ", STEP_IDS[masked_argmax(q_fake, mask)], "(eligible only)")

print("\nIntegration sketch:")
print(json.dumps(integration_sketch(), indent=2))

## Part 8 — Eval gate vs random baseline

Client MLOps: learned policy must beat random on simulator (smoke) and pass offline/shadow gates before production pin (`evals/thresholds.yaml` → `rl_smoke` / `rl_full`).

In [ ]:
from ml.fault_code_rl.eval_rl import evaluate_random, evaluate_q_table, evaluate_dqn_ckpt

env = DiagnosticMDP(seed=99)
rnd = evaluate_random(env, n=150, seed=2)
qmet = evaluate_q_table(q_result["Q"], DiagnosticMDP(seed=99), n=150)
dmet = evaluate_dqn_ckpt(ART / "dqn_policy.pt", DiagnosticMDP(seed=99), n=150)

report = {"random": rnd, "q_learning": qmet, "dqn": dmet}
print(json.dumps(report, indent=2))
(ART / "rl_eval_report.json").write_text(json.dumps(report, indent=2))

# simple smoke gate: at least one learned policy beats random success rate
delta_q = qmet["success_rate"] - rnd["success_rate"]
delta_d = dmet["success_rate"] - rnd["success_rate"]
print(f"Δ success Q={delta_q:+.3f} DQN={delta_d:+.3f}")
ok = max(delta_q, delta_d) >= 0.0  # weak smoke; tighten for client
print("SMOKE GATE:", "PASS" if ok else "FAIL (train longer)")

## Part 9 — CUDA & MLOps (aligned with vision stack)

| Concern | Vision (`fault_code_vision`) | RL (`fault_code_rl`) |
|---------|------------------------------|----------------------|
| Device helpers | `device_report`, `assert_cuda_for_client_train` | **Same** (re-exported) |
| GPU image | `docker/Dockerfile.ml` | **Same image** |
| Deps | `requirements-ml.txt` | **Same file** |
| Registry | `fault-code-ocr`, `fault-code-gan` | `diagnosis-step-bandit`, `diagnosis-session-dqn` |
| Eval floors | `vision_*` in `thresholds.yaml` | `rl_*` |
| Train CLI | `python -m ml.fault_code_vision.train_ocr` | `python -m ml.fault_code_rl.train` |

### Hardware guidance

| Algorithm | GPU needed? |
|-----------|-------------|
| Epsilon-greedy / UCB / Thompson / LinUCB | **No** (CPU) |
| Tabular Q-learning | **No** |
| DQN / future deep offline RL | **Yes (CUDA)** for client-scale train |

### Runtime integration (target)

```text
GraphRAG eligible steps ──► action mask
context (product, code, symptoms, steps_done)
        │
        ▼
  bandit / DQN policy (pinned registry alias)
        │
        ▼
  next step to agent UI
        │
        ▼
  outcome logger → offline train → eval → canary
```

In [ ]:
df = pd.DataFrame(mlops_checklist())
print(df.to_string(index=False))
df.to_csv(ART / "rl_mlops_checklist.csv", index=False)

# Show registry stubs
import yaml
reg = yaml.safe_load((ROOT / "models" / "registry.yaml").read_text())
rl_aliases = {k: v for k, v in reg["models"].items() if "diagnosis" in k or k.endswith("dqn") or "bandit" in k}
# filter rl-related
rl_aliases = {k: v for k, v in reg["models"].items() if k.startswith("diagnosis-")}
print("\nRegistry stubs:")
print(yaml.dump(rl_aliases, sort_keys=False))

summary = {
    "playbook": "notebooks/fault_code_rl_playbook.ipynb",
    "package": "ml/fault_code_rl",
    "recommended_first_algo": "contextual_bandit_LinUCB",
    "deep_rl_algo": "DQN_on_DiagnosticMDP",
    "cuda": "Required for DQN client trains; bandits/Q on CPU",
    "constraint": "action_mask=graph_eligible_CONFIRMS_only",
    "do_not": "replace_GraphRAG_with_unconstrained_RL",
}
(ART / "executive_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## Part 10 — Commands, references, client talking points

### Commands

```bash
pip install -r requirements-ml.txt

# Bandit bake-off
python -m ml.fault_code_rl.train --algo bandit --episodes 800 \
  --out notebooks/fault_code_rl_artifacts/bandit

# Tabular Q
python -m ml.fault_code_rl.train --algo q --episodes 1500 \
  --out notebooks/fault_code_rl_artifacts/q_policy.pt

# DQN on client CUDA
python -m ml.fault_code_rl.train --algo dqn --episodes 800 --require-cuda \
  --out models/artifacts/rl/dqn.pt

python -m ml.fault_code_rl.eval_rl \
  --q-table notebooks/fault_code_rl_artifacts/q_policy.npz \
  --dqn models/artifacts/rl/dqn.pt \
  --min-success-rate 0.35
```

### References

1. Sutton & Barto — *Reinforcement Learning: An Introduction* (MDP, Q-learning)
2. Auer et al. — UCB1 finite-time bandit analysis
3. Li et al. — *A Contextual-Bandit Approach to Personalized News Article Recommendation* (LinUCB, WWW 2010)
4. Mnih et al. — *Human-level control through deep RL* (DQN, Nature 2015)
5. Levine et al. — Offline RL tutorials / batch RL surveys
6. This repo LLMOps: `docs/sdd/09-PLATFORM-LLMOPS.md`

### Client talking points (30 seconds)

1. **Core diagnosis stays GraphRAG** — auditable, versioned knowledge.
2. **RL is optional optimization** of *which eligible step next*, using claim outcomes as reward.
3. We start with **contextual bandits + offline eval**, not wild online exploration.
4. **DQN + CUDA** is available for multi-step policies when data justifies it — **same GPU image** as vision training.
5. Safety: **action masks**, shadow mode, registry pins, canary rollback.

---

### Artifacts from this notebook

```text
notebooks/fault_code_rl_artifacts/
  bandit_curves.png
  q_learning_curve.png
  dqn_curves.png
  q_policy.npz
  dqn_policy.pt
  rl_eval_report.json
  rl_mlops_checklist.csv
  executive_summary.json
```

**Done when:** client sees theory + math + working bandit/Q/DQN, CUDA path documented, and a clear “when RL / when not” story integrated with GraphRAG.